# Round 2 — closing the validation → Kaggle gap

**Starting point.** The first submission (E3, EfficientNet-B0 fine-tuned without augmentation, h-flip TTA) scored **0.9749** on the public leaderboard although its validation accuracy was **0.9960**.

**Diagnosis.**
1. *Optimistic validation.* Every face photo was cut into face / 2 × eye / mouth / nose crops, and the ids are shuffled, so crops of the **same animal** land in both the training and the validation split. Validation therefore partly measures *recognising known animals*; the test set contains new animals. This also explains why the top validation error is CatEye ↔ DogEye (species), not a part confusion.
2. *Crop-geometry / quality shift.* Test crops are much less often exactly square and many are tiny and pixelated (Section 1.5 of the main notebook).
3. E3 shows the largest train/validation loss gap (0.0005 vs 0.015) — it memorised the training animals most.

**Plan.** Favour *robustness* over validation accuracy:
* **A.** Re-use the trained checkpoints: E4 (augmentation + label smoothing) alone, and ensembles of E3 + E4.
* **B.** Train a stronger backbone (**ConvNeXt-Tiny**, ImageNet-22k pre-training) with **heavier, test-like augmentation**: random down-scaling / pixelation, wider crop shapes and scales, blur, stronger colour jitter — then ensemble it with E4.

Each idea writes its own submission file to `outputs/` so they can be compared on the leaderboard (Task 6: compare model versions).

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

import ffc_lib as L

warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = Path("enge-707-2026-s-2-cat-and-dog-facial-feature-classification-ch")
OUT_DIR = Path("outputs")
SEED, IMG_SIZE, NUM_WORKERS = 42, 224, 4
CLASS_NAMES = ["CatEye", "DogEye", "CatFace", "DogFace", "CatMouth", "DogMouth", "CatNose", "DogNose"]
class_to_idx = json.load(open(OUT_DIR / "label_map.json"))
idx_to_class = {v: k for k, v in class_to_idx.items()}

TRAIN_CONVNEXT = True          # Part B (about 45-60 min on a laptop GPU)
CONVNEXT_EPOCHS = 8
CONVNEXT_BATCH = 32

L.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

# Same split as the main notebook (saved there) and the same test images
train_split = pd.read_csv(OUT_DIR / "train_split.csv")
val_split = pd.read_csv(OUT_DIR / "val_split.csv")
ds = L.discover_dataset(DATA_DIR, CLASS_NAMES)
test_df, sample_sub = ds["test_df"], ds["sample_sub"]
print(f"train {len(train_split)} | val {len(val_split)} | test {len(test_df)}")

_, eval_tf = L.build_transforms(IMG_SIZE, augment=False)
val_loader = L.make_loader(L.ImageDataset(val_split.filepath, val_split.y, eval_tf), 128, False, 2, SEED)
test_loader = L.make_loader(L.ImageDataset(test_df.filepath, None, eval_tf, ids=test_df.key), 128, False,
                            NUM_WORKERS, SEED, persistent=False)
crit = nn.CrossEntropyLoss()

In [ ]:
def write_submission(test_probs, name):
    # Same ids / order / columns as sample_submission.csv
    pred = test_probs.argmax(1)
    by_key = dict(zip(test_keys, pred))
    id_col, label_col = sample_sub.columns[:2]
    sub = sample_sub[[id_col]].copy()
    sub[label_col] = sub[id_col].map(lambda x: idx_to_class[int(by_key[L.norm_id(x)])])
    assert len(sub) == len(sample_sub) and sub[label_col].isin(CLASS_NAMES).all()
    path = OUT_DIR / f"submission_{name}.csv"
    sub.to_csv(path, index=False)
    return path

def load_trained(name, arch):
    m = L.build_model(arch, len(CLASS_NAMES), pretrained=False).to(device)
    m.load_state_dict(torch.load(OUT_DIR / f"{name}_best.pt", map_location=device))
    return m.eval()

val_probs, test_probs, rows = {}, {}, []
def register(name, model):
    global test_keys
    _, _, vp, vy = L.evaluate(model, val_loader, crit, device, tta=True)
    test_keys, tp = L.predict(model, test_loader, device, tta=True)
    val_probs[name], test_probs[name] = vp, tp
    np.save(OUT_DIR / f"test_probs_{name}.npy", tp)
    rows.append(dict(model=name, val_acc=accuracy_score(vy, vp.argmax(1)),
                     val_macro_f1=f1_score(vy, vp.argmax(1), average="macro"),
                     test_pred_agree_with_E3=None))
    return vy

## A. Re-use existing checkpoints: E4 alone and E3 + E4 ensembles

In [ ]:
t0 = time.time()
for name in ["effb0_ft_noaug", "effb0_ft"]:
    m = load_trained(name, "efficientnet_b0")
    val_y = register(name, m)
    del m; torch.cuda.empty_cache()

def ensemble(names, weights=None):
    w = np.ones(len(names)) if weights is None else np.asarray(weights, float)
    v = sum(wi * val_probs[n] for wi, n in zip(w, names)) / w.sum()
    t = sum(wi * test_probs[n] for wi, n in zip(w, names)) / w.sum()
    return v, t

v, t = ensemble(["effb0_ft_noaug", "effb0_ft"])
val_probs["ens_e3_e4"], test_probs["ens_e3_e4"] = v, t
rows.append(dict(model="ens_e3_e4", val_acc=accuracy_score(val_y, v.argmax(1)),
                 val_macro_f1=f1_score(val_y, v.argmax(1), average="macro")))

for name in ["effb0_ft", "ens_e3_e4"]:
    print("wrote", write_submission(test_probs[name], name))

ref = test_probs["effb0_ft_noaug"].argmax(1)
res = pd.DataFrame(rows).drop(columns="test_pred_agree_with_E3", errors="ignore")
res["test_agreement_with_E3"] = [ (test_probs[n].argmax(1) == ref).mean() for n in res.model ]
display(res.round(4))
print(f"Part A took {(time.time()-t0)/60:.1f} min")

`test_agreement_with_E3` shows how many test predictions change compared with the submitted model — with a 2.5-point gap to fix, a useful candidate must disagree on at least a few percent of test images.

## B. ConvNeXt-Tiny + test-like augmentation

In [ ]:
strong_tf, _ = L.build_strong_transforms(IMG_SIZE)
print(strong_tf)

# Show what the new augmentation does
ex = train_split.groupby("label").head(1).head(4)
fig, axes = plt.subplots(4, 7, figsize=(14, 8.4))
from PIL import Image
for r, (_, row) in enumerate(ex.iterrows()):
    with Image.open(row.filepath) as im:
        img = im.convert("RGB")
    axes[r, 0].imshow(L.denormalize(eval_tf(img))); axes[r, 0].set_title(row.label, fontsize=9)
    for c in range(1, 7):
        axes[r, c].imshow(L.denormalize(strong_tf(img))); axes[r, c].set_title(f"strong aug #{c}", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "18_strong_augmentation.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
if TRAIN_CONVNEXT:
    L.seed_everything(SEED)
    train_loader = L.make_loader(L.ImageDataset(train_split.filepath, train_split.y, strong_tf),
                                 CONVNEXT_BATCH, True, NUM_WORKERS, SEED, drop_last=True)
    cfg = L.TrainConfig("convnext_tiny_strong", "convnext_tiny.fb_in22k_ft_in1k", epochs=CONVNEXT_EPOCHS,
                        lr=4e-4, backbone_lr_mult=0.25, weight_decay=0.05, label_smoothing=0.1,
                        drop_rate=0.3, patience=3, warmup_epochs=1)
    finished = all((OUT_DIR / f"{cfg.name}{x}").exists() for x in ("_best.pt", "_history.csv", "_summary.json"))
    if finished:
        print("Loading finished ConvNeXt run")
        cnx = load_trained(cfg.name, cfg.arch)
        hist = pd.read_csv(OUT_DIR / f"{cfg.name}_history.csv")
    else:
        cnx, hist, summ = L.fit(cfg, train_loader, val_loader, len(CLASS_NAMES), device, OUT_DIR)
    display(hist.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(hist.epoch, hist.train_loss, "o-", label="train"); axes[0].plot(hist.epoch, hist.val_loss, "o-", label="val")
    axes[1].plot(hist.epoch, hist.train_acc, "o-", label="train"); axes[1].plot(hist.epoch, hist.val_acc, "o-", label="val")
    axes[0].set_title("ConvNeXt-Tiny (strong aug) - loss"); axes[1].set_title("accuracy")
    for ax in axes: ax.legend(); ax.set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "19_convnext_curves.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
if TRAIN_CONVNEXT:
    register("convnext_tiny_strong", cnx.eval())
    del cnx; torch.cuda.empty_cache()
    for names, tag in [(["convnext_tiny_strong", "effb0_ft"], "ens_cnx_e4"),
                       (["convnext_tiny_strong", "effb0_ft", "effb0_ft_noaug"], "ens_cnx_e4_e3")]:
        v, t = ensemble(names)
        val_probs[tag], test_probs[tag] = v, t
        rows.append(dict(model=tag, val_acc=accuracy_score(val_y, v.argmax(1)),
                         val_macro_f1=f1_score(val_y, v.argmax(1), average="macro")))
    for name in ["convnext_tiny_strong", "ens_cnx_e4", "ens_cnx_e4_e3"]:
        print("wrote", write_submission(test_probs[name], name))

res = pd.DataFrame(rows).drop(columns="test_pred_agree_with_E3", errors="ignore")
res["test_agreement_with_E3"] = [(test_probs[n].argmax(1) == ref).mean() for n in res.model]
res["submission_file"] = ["submission_" + n + ".csv" if (OUT_DIR / f"submission_{n}.csv").exists() else
                          ("submission.csv (already submitted)" if n == "effb0_ft_noaug" else "") for n in res.model]
res.to_csv(OUT_DIR / "round2_results.csv", index=False)
display(res.round(4))

## Leaderboard log
Record every Kaggle score here (and screenshot each submission for the report).

| Submission file | Model | Validation acc | Public LB |
|---|---|---|---|
| submission.csv | E3 EfficientNet-B0, no aug, TTA | 0.9960 | **0.9749** |
| submission_convnext_tiny_strong.csv | ConvNeXt-Tiny, strong aug | see table | |
| submission_ens_cnx_e4.csv | ConvNeXt + E4 average | see table | |
| submission_effb0_ft.csv | E4 EfficientNet-B0, aug + label smoothing | see table | |
| submission_ens_e3_e4.csv | E3 + E4 average | see table | |